In [ ]:
# ===== CONFIGURE HERE (Change based on your setup) =====
# For Google Colab:
# BASE_URL = "/content/drive/MyDrive/experiments"

# For Local (Windows):
BASE_URL = "D:/MPHIL_CODES/MPHIL_MAIN_REPO/experiments"

# For Local (Mac/Linux):
# BASE_URL = "/Users/yourname/experiments"

from pathlib import Path
BASE_PATH = Path(BASE_URL)
ARCADE_PATH = BASE_PATH / 'datasets' / 'ARCADE'

# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    pass

# YOLOv8 Training on ARCADE Dataset - Google Colab

Train YOLOv8-seg model on ARCADE coronary artery segmentation dataset

**Dataset location:** `/content/drive/MyDrive/MPhil Research/Datasets/ARCADE/`

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted")

In [ ]:
# ===== TRAINING CONFIGURATION (Change these values) =====
EPOCHS = 50              # Number of training epochs
BATCH_SIZE = 8           # Batch size
LEARNING_RATE = 1e-3    # Learning rate
IMAGE_SIZE = 640        # Input image size
# =====================================================

print(f'Configuration:')
print(f'  EPOCHS: {EPOCHS}')
print(f'  BATCH_SIZE: {BATCH_SIZE}')
print(f'  LEARNING_RATE: {LEARNING_RATE}')
print(f'  IMAGE_SIZE: {IMAGE_SIZE}x{IMAGE_SIZE}')


## 2. Install Dependencies

In [ ]:
!pip install -q ultralytics torch pandas

## 3. Setup Paths

In [ ]:
import os
import json
import shutil
from pathlib import Path

DATASETS_DIR = Path('/content/drive/MyDrive/MPhil Research/Datasets')
arcade_path = DATASETS_DIR / 'ARCADE'
yolo_dataset = DATASETS_DIR / 'yolo_arcade_dataset'
output_dir = DATASETS_DIR / 'yolov8_outputs'

output_dir.mkdir(exist_ok=True)

print(f"Dataset: {arcade_path}")
print(f"YOLO format: {yolo_dataset}")
print(f"Output: {output_dir}")
print(f"\n✓ ARCADE exists: {arcade_path.exists()}")

## 4. Check GPU

In [ ]:
import torch

cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")

if cuda_available:
    device = 0
    batch_size = 8
    epochs = 100
    print(f"✓ Using GPU (batch={batch_size}, epochs={epochs})")
else:
    device = 'cpu'
    batch_size = 2
    epochs = 10
    print(f"Using CPU (batch={batch_size}, epochs={epochs})")

## 5. Convert ARCADE to YOLO Format

In [ ]:
if yolo_dataset.exists():
    print(f"✓ YOLO dataset already exists")
else:
    print(f"Converting ARCADE to YOLO format...\n")
    
    class ArcadeToYOLO:
        def __init__(self, arcade_root, output_root):
            self.arcade_root = Path(arcade_root)
            self.output_root = Path(output_root)
            self.output_root.mkdir(parents=True, exist_ok=True)

        def convert(self):
            for split in ['train', 'val']:
                print(f"Processing {split}...")
                self._convert_split(split)
            self._create_yaml()
            print(f"\n✓ Conversion complete!")

        def _convert_split(self, split):
            (self.output_root / split / 'images').mkdir(parents=True, exist_ok=True)
            (self.output_root / split / 'labels').mkdir(parents=True, exist_ok=True)
            
            images_dir = self.arcade_root / 'stenosis' / split / 'images'
            ann_file = self.arcade_root / 'stenosis' / split / 'annotations' / f'{split}.json'

            with open(ann_file, 'r') as f:
                coco_data = json.load(f)

            sorted_cats = sorted(coco_data['categories'], key=lambda x: x['id'])
            cat_map = {cat['id']: idx for idx, cat in enumerate(sorted_cats)}

            for img_info in coco_data['images']:
                img_id = img_info['id']
                img_file = img_info['file_name']
                
                src = images_dir / img_file
                dst = self.output_root / split / 'images' / img_file
                if src.exists():
                    shutil.copy(src, dst)

                anns = [a for a in coco_data['annotations'] if a['image_id'] == img_id]
                if not anns:
                    continue

                lines = []
                for ann in anns:
                    cls_id = cat_map[ann['category_id']]
                    if 'segmentation' in ann and ann['segmentation']:
                        seg = ann['segmentation'][0]
                        h, w = img_info['height'], img_info['width']
                        norm = [coord / (w if i % 2 == 0 else h) for i, coord in enumerate(seg)]
                        line = str(cls_id) + ' ' + ' '.join(f'{c:.6f}' for c in norm)
                        lines.append(line)

                if lines:
                    label_file = self.output_root / split / 'labels' / (Path(img_file).stem + '.txt')
                    with open(label_file, 'w') as f:
                        f.write('\n'.join(lines))
            
            print(f"  ✓ {len(coco_data['images'])} images")

        def _create_yaml(self):
            ann_file = self.arcade_root / 'stenosis' / 'train' / 'annotations' / 'train.json'
            with open(ann_file, 'r') as f:
                coco_data = json.load(f)

            sorted_cats = sorted(coco_data['categories'], key=lambda x: x['id'])
            names = {idx: cat['name'] for idx, cat in enumerate(sorted_cats)}

            yaml = f"""path: {self.output_root}
train: train/images
val: val/images

nc: {len(names)}
names:
"""
            for idx in sorted(names.keys()):
                yaml += f"  {idx}: {names[idx]}\n"

            with open(self.output_root / 'data.yaml', 'w') as f:
                f.write(yaml)

    converter = ArcadeToYOLO(arcade_path, yolo_dataset)
    converter.convert()

print(f"\n✓ YOLO dataset ready at: {yolo_dataset}")

## 6. Load Model

In [ ]:
from ultralytics import YOLO

print("Loading YOLOv8m-seg...")
model = YOLO('yolov8m-seg.pt')
print("✓ Model loaded")

## 7. Train Model

In [ ]:
DATA_YAML = yolo_dataset / 'data.yaml'

print(f"\nStarting training (epochs={epochs}, batch={batch_size})...\n")

results = model.train(
    data=str(DATA_YAML),
    epochs=epochs,
    imgsz=512,
    batch=batch_size,
    device=device,
    patience=5,
    save=True,
    project=str(output_dir),
    name='arcade_yolov8m_seg',
    exist_ok=False,
    workers=0
)

print(f"\n✓ Training complete!")

## 8. Show Results

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

runs_dir = output_dir / 'arcade_yolov8m_seg'
results_csv = runs_dir / 'results.csv'

if results_csv.exists():
    df = pd.read_csv(results_csv)
    print(f"\n{'='*60}")
    print(f"Training complete! Epochs: {len(df)}")
    print(f"{'='*60}")
    
    if 'metrics/mAP50(B)' in df.columns:
        best_idx = df['metrics/mAP50(B)'].idxmax()
        print(f"\nBest mAP50: {df['metrics/mAP50(B)'].iloc[best_idx]:.4f} (epoch {int(df['epoch'].iloc[best_idx])})")
    
    # Plot results
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    if 'train/box_loss' in df.columns:
        axes[0, 0].plot(df.index, df['train/box_loss'], label='train', marker='.')
        if 'val/box_loss' in df.columns:
            axes[0, 0].plot(df.index, df['val/box_loss'], label='val', marker='.')
        axes[0, 0].set_title('Box Loss')
        axes[0, 0].legend()
        axes[0, 0].grid(alpha=0.3)
    
    if 'train/seg_loss' in df.columns:
        axes[0, 1].plot(df.index, df['train/seg_loss'], label='train', marker='.')
        if 'val/seg_loss' in df.columns:
            axes[0, 1].plot(df.index, df['val/seg_loss'], label='val', marker='.')
        axes[0, 1].set_title('Seg Loss')
        axes[0, 1].legend()
        axes[0, 1].grid(alpha=0.3)
    
    if 'train/cls_loss' in df.columns:
        axes[1, 0].plot(df.index, df['train/cls_loss'], label='train', marker='.')
        if 'val/cls_loss' in df.columns:
            axes[1, 0].plot(df.index, df['val/cls_loss'], label='val', marker='.')
        axes[1, 0].set_title('Class Loss')
        axes[1, 0].legend()
        axes[1, 0].grid(alpha=0.3)
    
    if 'metrics/mAP50(B)' in df.columns:
        axes[1, 1].plot(df.index, df['metrics/mAP50(B)'], label='mAP50', marker='.', color='green')
        axes[1, 1].set_title('mAP50')
        axes[1, 1].legend()
        axes[1, 1].grid(alpha=0.3)
        axes[1, 1].set_ylim([0, 1])
    
    plt.tight_layout()
    plt.savefig(runs_dir / 'results.png', dpi=100)
    print(f"\n✓ Results saved to: {runs_dir}")
    plt.show()

In [ ]:
print("\n" + "="*80)
print("TRAINING ACCURACY & PERFORMANCE METRICS")
print("="*80)

if results_csv.exists():
    df = pd.read_csv(results_csv)
    
    # Last epoch metrics
    last_epoch = len(df)
    print(f"\n📊 Last Epoch ({last_epoch}) Metrics:")
    print(f"  Box Loss:       {df['train/box_loss'].iloc[-1]:.4f}")
    print(f"  Seg Loss:       {df['train/seg_loss'].iloc[-1]:.4f}")
    print(f"  Class Loss:     {df['train/cls_loss'].iloc[-1]:.4f}")
    
    if 'metrics/mAP50(B)' in df.columns:
        print(f"  Val mAP50:      {df['metrics/mAP50(B)'].iloc[-1]:.4f}")
    if 'metrics/mAP50-95(B)' in df.columns:
        print(f"  Val mAP50-95:   {df['metrics/mAP50-95(B)'].iloc[-1]:.4f}")
    
    # Best epoch
    print(f"\n🏆 Best Performance:")
    if 'metrics/mAP50(B)' in df.columns:
        best_idx = df['metrics/mAP50(B)'].idxmax()
        print(f"  Best Epoch:     {int(df['epoch'].iloc[best_idx])}")
        print(f"  Best mAP50:     {df['metrics/mAP50(B)'].iloc[best_idx]:.4f}")
    
    # Training progress
    print(f"\n📈 Training Progress:")
    print(f"  Box Loss (start → end):     {df['train/box_loss'].iloc[0]:.4f} → {df['train/box_loss'].iloc[-1]:.4f}")
    print(f"  Seg Loss (start → end):     {df['train/seg_loss'].iloc[0]:.4f} → {df['train/seg_loss'].iloc[-1]:.4f}")
    print(f"  Class Loss (start → end):   {df['train/cls_loss'].iloc[0]:.4f} → {df['train/cls_loss'].iloc[-1]:.4f}")
    
    print(f"\n{'='*80}")

## 9. Validate

In [ ]:
print("Validating model...")
model.val(data=str(DATA_YAML), device=device)
print("✓ Validation complete!")

## 10. Summary

In [ ]:
best_model = runs_dir / 'weights' / 'best.pt'
print(f"\n✓ Training complete!")
print(f"Best model: {best_model}")
print(f"Results: {runs_dir}")
print(f"\n🎉 YOLOv8 model trained on ARCADE!")